Install Dependencies

In [ ]:
!pip install transformers langchain langchain-community

Enable LangSmith Tracing

In [54]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_key_here"

Load HuggingFace Model

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=512
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

Define Job Description & Resumes

In [56]:
from pydantic import BaseModel, Field

job_description = """
Data Scientist role:
- Python, SQL, Machine Learning
- Pandas, NumPy, Scikit-learn
- Deep Learning (TensorFlow/PyTorch)
- Tableau or Power BI
- 2+ years experience
"""

strong_resume = """
3 years experience in Data Science.
Skills: Python, SQL, Machine Learning, Deep Learning
Tools: Pandas, NumPy, Scikit-learn, TensorFlow, Tableau
"""

average_resume = """
1.5 years experience in analytics.
Skills: Python, SQL
Tools: Pandas, Excel
"""

weak_resume = """
Fresher with basic computer knowledge.
Skills: MS Word, Communication
"""

class ResumeData(BaseModel):
    skills: list[str] = Field(description="List of skills mentioned in the resume")
    tools: list[str] = Field(description="List of tools mentioned in the resume")
    experience: str = Field(description="Years of experience mentioned in the resume")

Create Prompt Templates

In [57]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ResumeData)

extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of strings (e.g., ["Python", "SQL"])
- "tools": a JSON array of strings (e.g., ["Pandas", "NumPy"])
- "experience": a string representing years of experience (e.g., "3 years")

Do NOT include any other text, explanation, or conversational filler.
Fill in the lists and the experience string based on the resume.

Example output format:
```json
{{
  "skills": ["Python", "SQL"],
  "tools": ["Pandas", "NumPy", "Scikit-learn"],
  "experience": "3 years"
}}
```

Resume:
{resume}
"""
)

match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
Compare resume with job description.

Return:
Matched Skills:
Missing Skills:
Experience Match (Yes/No):

Resume:
{resume_data}

Job:
{job_description}
"""
)

sense_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Score candidate (0-100) based on:

- Skills match (50%)
- Tools match (30%)
- Experience (20%)

Rules:
- Strong → 80+
- Medium → 50-79
- Weak → <50

Return:
Score: number

Match Data:
{match_data}
"""
)

explain_prompt = PromptTemplate(
    input_variables=["match_data", "score"],
    template="""
Explain the score clearly:

- matched skills
- missing skills
- experience gap

Score: {score}

Match Data:
{match_data}
"""
)

Build LCEL Chains

In [58]:
from langchain_core.output_parsers import StrOutputParser

extraction_chain = extract_prompt | llm | StrOutputParser()
matching_chain = match_prompt | llm
scoring_chain = sense_prompt | llm
explanation_chain = explain_prompt | llm

Define Pipeline Function

In [59]:
import json
from pydantic import ValidationError

def run_pipeline(resume, label):
    print(f"\n===== {label} ====")

    # Step 1: Extraction
    raw_extracted_text = extraction_chain.invoke({"resume": resume})
    print("\n[Raw LLM Extraction Output]")
    print(raw_extracted_text)

    try:
        extracted_json_data = json.loads(raw_extracted_text)
        extracted_pydantic_obj = ResumeData.model_validate(extracted_json_data)
        extracted_json_str = extracted_pydantic_obj.model_dump_json(indent=2)
        print("\n[Validated Extraction]")
        print(extracted_json_str)

    except json.JSONDecodeError as e:
        print(f"\n[Extraction Error] Failed to decode JSON: {e}")
        print(f"Raw output was: {raw_extracted_text}")
        return
    except ValidationError as e:
        print(f"\n[Extraction Error] Pydantic validation failed: {e}")
        print(f"Parsed JSON was: {extracted_json_data}")
        return

    # Step 2: Matching
    match = matching_chain.invoke({
        "resume_data": extracted_json_str,
        "job_description": job_description
    })
    print("\n[Matching]")
    print(match)

    # Step 3: Scoring
    score = scoring_chain.invoke({"match_data": match})
    print("\n[Score]")
    print(score)

    # Step 4: Explanation
    explanation = explanation_chain.invoke({
        "match_data": match,
        "score": score
    })
    print("\n[Explanation]")
    print(explanation)

Run It

In [60]:
run_pipeline(strong_resume, "STRONG CANDIDATE")


===== STRONG CANDIDATE ====


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Raw LLM Extraction Output]

Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of strings (e.g., ["Python", "SQL"])
- "tools": a JSON array of strings (e.g., ["Pandas", "NumPy"])
- "experience": a string representing years of experience (e.g., "3 years")

Do NOT include any other text, explanation, or conversational filler.
Fill in the lists and the experience string based on the resume.

Example output format:
```json
{
  "skills": ["Python", "SQL"],
  "tools": ["Pandas", "NumPy", "Scikit-learn"],
  "experience": "3 years"
}
```

Resume:

3 years experience in Data Science.
Skills: Python, SQL, Machine Learning, Deep Learning
Tools: Pandas, NumPy, Scikit-learn, TensorFlow, Tableau



[Extraction Error] Failed to decode JSON: Expecting value: line 2 column 1 (char 1)
Raw output was: 
Based on the following resume, extract the candidate's skills, to

In [61]:
run_pipeline(average_resume, "AVERAGE CANDIDATE")


===== AVERAGE CANDIDATE ====


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Raw LLM Extraction Output]

Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of strings (e.g., ["Python", "SQL"])
- "tools": a JSON array of strings (e.g., ["Pandas", "NumPy"])
- "experience": a string representing years of experience (e.g., "3 years")

Do NOT include any other text, explanation, or conversational filler.
Fill in the lists and the experience string based on the resume.

Example output format:
```json
{
  "skills": ["Python", "SQL"],
  "tools": ["Pandas", "NumPy", "Scikit-learn"],
  "experience": "3 years"
}
```

Resume:

1.5 years experience in analytics.
Skills: Python, SQL
Tools: Pandas, Excel



[Extraction Error] Failed to decode JSON: Expecting value: line 2 column 1 (char 1)
Raw output was: 
Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with 

In [62]:
run_pipeline(weak_resume, "WEAK CANDIDATE")


===== WEAK CANDIDATE ====


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Raw LLM Extraction Output]

Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of strings (e.g., ["Python", "SQL"])
- "tools": a JSON array of strings (e.g., ["Pandas", "NumPy"])
- "experience": a string representing years of experience (e.g., "3 years")

Do NOT include any other text, explanation, or conversational filler.
Fill in the lists and the experience string based on the resume.

Example output format:
```json
{
  "skills": ["Python", "SQL"],
  "tools": ["Pandas", "NumPy", "Scikit-learn"],
  "experience": "3 years"
}
```

Resume:

Fresher with basic computer knowledge.
Skills: MS Word, Communication



[Extraction Error] Failed to decode JSON: Expecting value: line 2 column 1 (char 1)
Raw output was: 
Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the fo

Debug Case

In [63]:
bug_resume = "I worked with data."

run_pipeline(bug_resume, "BUG CASE")


===== BUG CASE ====


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Raw LLM Extraction Output]

Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of strings (e.g., ["Python", "SQL"])
- "tools": a JSON array of strings (e.g., ["Pandas", "NumPy"])
- "experience": a string representing years of experience (e.g., "3 years")

Do NOT include any other text, explanation, or conversational filler.
Fill in the lists and the experience string based on the resume.

Example output format:
```json
{
  "skills": ["Python", "SQL"],
  "tools": ["Pandas", "NumPy", "Scikit-learn"],
  "experience": "3 years"
}
```

Resume:
I worked with data.


[Extraction Error] Failed to decode JSON: Expecting value: line 2 column 1 (char 1)
Raw output was: 
Based on the following resume, extract the candidate's skills, tools, and years of experience.
Your output MUST be a JSON object with the following keys and types:
- "skills": a JSON array of 

AI Resume Screening System

Pipeline:
Resume → Extraction → Matching → Scoring → Explanation

Technologies Used:
- LangChain
- HuggingFace (flan-t5-base)
- PromptTemplate
- LCEL (|)
- invoke()

Features:
- Skill extraction
- Resume-job matching
- Scoring (0–100)
- Explanation generation

Debug Case:
The model incorrectly assumed skills for vague input.
Fix: Added strict instruction "Do NOT assume missing skills"

LangSmith:
Tracing enabled using LANGCHAIN_TRACING_V2